# Training
The smoke test uses 32 training and 16 validation images for one epoch. It checks training and checkpointing, not model performance.

In [ ]:
from pathlib import Path
import os
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "configs/cpu.yaml").exists():
    raise RuntimeError("Open this notebook from the project root or notebooks directory")
os.chdir(ROOT)
import yaml
from ml.runtime import configure
config = yaml.safe_load(Path("configs/cpu.yaml").read_text())
configure(config["seed"], config["threads"])

In [ ]:
from ml.training.trainer import train
RUN_SMOKE_TEST = False  # Set True to run the smoke test.
if RUN_SMOKE_TEST:
    history = train(config, smoke=True)
else:
    print("Smoke test is opt-in; no training started.")

## Full training
Get approval before starting full training. The default is up to five head-training epochs with early stopping; fine-tuning is off. Validation ROC-AUC selects the checkpoint. The decision threshold is fixed at 0.5. Test images are not used.

In [ ]:
FULL_TRAINING_APPROVED = False
RESUME_PATH = None  # Use "artifacts/checkpoints/run/last.pt" to resume.
if FULL_TRAINING_APPROVED:
    history = train(config, smoke=False, approved_full_training=True, resume=RESUME_PATH)
else:
    print("Full training awaits explicit approval.")

## Resume training
Set RESUME_PATH in the training cell above. Checkpoints save model, optimizer, scheduler, random state, configuration, split fingerprint, and history at epoch boundaries. Resuming repeats any unfinished epoch and requires the same configuration. The app does not accept smoke checkpoints.

In [ ]:
import json
from IPython.display import display, Image
history_path = Path(config["checkpoint_dir"]) / ("run" if FULL_TRAINING_APPROVED else "smoke") / "history.json"
if history_path.exists():
    import pandas as pd
    display(pd.json_normalize(json.loads(history_path.read_text())))
    display(Image(filename=str(history_path.parent / "training_curves.png")))